In [ ]:
import pandas as pd
import pickle
import os
import matplotlib.pyplot as plt
import seaborn as sns

## Lecture des données contenues dans le fichier pickle

In [ ]:
source_path = "."
tbl_df_path = os.path.join(source_path, "post_rehydrated.pickle")

## Exploration des données de la base tabulaire

In [ ]:
def read_pickle(df_path):
    """
    Permit to read a pickle file.
    Require package pickle
    """
    if not isinstance(df_path, str):
        raise TypeError("The dataframe path must be a string")
    with open(df_path, "rb") as f:
        data = pickle.load(f)
    return data

In [ ]:
tbl_df_path = "post_rehydrated.pickle"
tbl_df = read_pickle(tbl_df_path)

In [ ]:
tbl_df.head()

In [ ]:
tbl_df = pd.DataFrame(tbl_df)
tbl_df.columns

In [ ]:
tbl_df["account_followers"]

Cette première vue de la base de données m'emmène à me demander si les variables suivantes ne seraient-elles pas pertinentes : 

* `source_pf_account_id`
    Cette variable permet d’identifier de manière unique chaque compte source à l'origine du post. Cela peut être très utile pour analyser le comportement du compte et repérer des comptes inauthentiques ou automatisés. Si un même compte partage de manière répétée des images manipulées ou douteuses, ou si un compte n'a pas une histoire cohérente (comme un grand nombre de posts récents ou suspects), cela peut être un signe qu'il est utilisé dans des campagnes de désinformation. L’analyse de l’historique de publication d'un compte permet de repérer les patterns suspects de diffusion de contenu manipulé.

* `source_account_name`
    Le nom du compte peut fournir un indice sur sa légitimité. Les comptes qui utilisent des noms très génériques, comme "Compte Officiel" ou "Média2025", sont souvent associés à des comptes automatisés ou à des tentatives de manipulation. En outre, l’analyse du changement de nom d’un compte peut aussi signaler des tentatives de manipulation, par exemple un changement soudain de nom pour effacer l’historique d’un compte malveillant.

* `source_account_screen_name`
    Le nom d’écran (souvent appelé handle) est une autre forme d'identification unique d’un compte. Comme pour le nom du compte, un handle suspect ou générique (ex. : "news_2024_") pourrait indiquer un compte automatisé. Aussi, l’analyse des modifications du nom d’écran au fil du temps peut aider à repérer des comptes qui essaient de masquer leur activité, ce qui est souvent le cas dans des stratégies de désinformation. Le tuteur avait parlé des bots russes qui utilisaient ce genre d'handles génériques.

* `source_account_profile_picture`
    L’image de profil du compte peut être un autre indicateur de la légitimité du compte. (pas trellement pertinent mais bof)

* `source_post_id`  
  Cette variable permet d'identifier de manière unique chaque post. Si une même image est associée à plusieurs `source_post_id`, cela pourrait suggérer qu’elle a été *repartagée* ou *réutilisée dans un contexte trompeur*. Cela peut être utile pour détecter les comportements automatisés ou les tentatives de manipulation de l'image.

* `join_post_post_type`  
  Le type de post (par exemple, *original*, *repost*, *reshare*) peut être un indicateur de la manière dont l'image est utilisée. Les images *repartagées* ou *réutilisées* dans un but de *désinformation* sont souvent suspectes. Cette variable permet de mieux comprendre si l'image est manipulée dans différents contextes.

* `source_account_registered_at`  
  Si un compte est créé *juste avant ou après un post* contenant une image suspecte, cela peut être un signe de *comportement inauthentique*, comme des comptes automatisés créés pour publier du contenu manipulé ou de la *désynformation*.

* `source_account_description`  
  Les *descriptions vagues* ou *génériques* peuvent être associées à des comptes inauthentiques, souvent utilisés pour propager de la désinformation ou des images manipulées. Un compte avec une *description floue* peut indiquer un *compte automatisé* ou un *faux profil*, souvent utilisés dans des campagnes de manipulation.

* `source_date`  
  La *date du post* est cruciale pour comprendre le *timing* de l'image. Des posts avec des *intervalles de temps anormaux* entre eux ou *publiés rapidement* dans des périodes spécifiques (comme pendant des événements ou crises) peuvent être un indicateur de *manipulation intentionnelle* ou de *désinformation*. L'analyse des dates aide à détecter les tentatives de manipulation par l'intermédiaire des images.

* `document_image_list`  
  Cette colonne est directement liée aux images elles-mêmes. Si l'image est associée à plusieurs documents, ou si une même image apparaît dans plusieurs fichiers, cela peut suggérer que l'image a été utilisée dans des contextes *multiples* ou *manipulée* pour diffuser un message particulier. De plus, en analysant le contenu de `document_image_list`, tu peux rechercher des motifs inhabituels, comme la réutilisation d'une image dans des contextes différents sans explication claire.

* `source_post_engagements` et `post_engagements`  
  Les *engagements* (likes, commentaires, partages) peuvent fournir des indices sur l'authenticité de l'image. Des *engagements artificiels* ou anormaux, comme un grand nombre de réactions en peu de temps ou des *réactions suspectes*, peuvent signaler une image manipulée ou propagée de manière intentionnelle. Analyser les engagements peut aider à repérer les images manipulées ou utilisées de manière stratégique.

* `source_post_reactions` et `post_reactions`  
  Les types de *réactions* (positives ou négatives) sur un post peuvent en dire beaucoup sur l'authenticité d'une image. Un *grand nombre de réactions négatives*, comme des critiques ou des signalements de manipulation, pourrait indiquer qu'une image est perçue comme fausse ou trompeuse. Cette variable aide à évaluer la perception publique de l'image et à repérer des anomalies.

* `source_account_followers` et `account_followers`  
  Un nombre élevé de *followers* mais un faible *engagement* peut être un signe de *compte automatisé* ou de *fausse popularité*. Ces comptes sont souvent utilisés pour manipuler l'audience, en particulier lorsqu'ils sont associés à des images inauthentiques. L'analyse des followers permet de détecter les comptes suspects créés spécifiquement pour diffuser des images modifiées.

* `post_created_at`  
  L'heure et la date de création du post peuvent permettre d'identifier des images partagées à des moments clés pour manipuler l'opinion publique (par exemple, pendant des événements médiatiques spécifiques). L'analyse du timing de la publication peut également détecter des comportements automatisés, comme les images publiées *simultanément sur plusieurs plateformes*.

* `post_lang`  
  Cette variable peut aider à repérer les tentatives de manipulation de contenu dans *différents contextes culturels et linguistiques*. Par exemple, une image utilisée de manière trompeuse dans un *autre langage ou culture* peut être un signal de désinformation. L'analyse des *langues des posts* permet de détecter des images propagées dans un but de manipulation à travers différentes audiences.

* `image_name` et `source_image_name`  
  Le nom de l'image peut fournir des indices sur son origine ou son utilisation. Si une même *image est associée à des noms similaires* ou est réutilisée dans plusieurs contextes, cela peut être un signe de manipulation. Des noms d'image génériques ou *automatiquement générés* sont souvent associés à des images créées ou partagées par des *systèmes automatisés*.

En combinant ces variables, on pourrait identifier des comportements suspects, des motifs récurrents et des anomalies dans la manière dont les images sont utilisées, ce qui te permettra de mieux détecter les images inauthentiques et manipulées.


## Verification de `join_post_post_id`


In [ ]:
tbl_df["join_post_post_type"].unique()

In [ ]:
# tbl_df['document_image_list'].unique()
print(tbl_df["image_name"].dtype)  # type object

# conversion en chaine de caractères
tbl_df["image_name"] = tbl_df["image_name"].astype(str)

# ffichage des valeurs uniques
unique_values = tbl_df["image_name"].unique()
print(unique_values)

On voit ici qu'on a les labels des images. On peut donc associer chaque image à un post, donc à un compte.

In [ ]:
print(tbl_df["pf_account_id"].unique())

`007`, `zz` etc .. qui se repetent

In [ ]:
tbl_df["account_registered_at"].sort_values(ascending=False)

In [ ]:
tbl_df["source_pf_account_id"].unique()

In [ ]:
# nombre de suivis
display(tbl_df["account_following"].describe())

In [ ]:
# nombre de followers
tbl_df["account_followers"].describe()

In [ ]:
# heure des posts (retweets, reshares, posts originaux)
tbl_df["post_created_at"]

In [ ]:
# nombre de repartage d'un post
tbl_df["post_shares"]

In [ ]:
#
df[]

### Quelques analyses

In [ ]:
test = tbl_df.dropna(subset=["image_name"])
test["image_name"] = test["image_name"].astype(str)
x = test.groupby("image_name").size().reset_index(name="numb")

x = x.loc[x["numb"] > 1]
x

In [ ]:
# images réutilisées
def how_many_images_have_been_reused():
    """
    Permet de voir les images réutilisées
    (par exemple les retweets, partages) et
    les comptes qui les ont partagées, ainsi
    que le nombre de fois qu'elles ont été partagées.
    """

    # On group by 'image_name' et 'source_account_name'
    #  pour voir les comptes ayant partagé les mêmes images
    tbl_df_clean = tbl_df.dropna(
        subset=["image_name", "source_account_name", "source_pf_account_id"]
    )
    grouped = (
        tbl_df_clean.groupby(
            ["image_name", "source_account_name", "source_pf_account_id"]
        )
        .size()
        .reset_index(name="number_of_retweets")
    )

    print(grouped)

    grouped = grouped[grouped["number_of_retweets"] > 1]
    # grouped = grouped[grouped['source_account_name'].str.lower().str.contains('nicola', na=False)]
    results = {}

    for _, row in grouped.iterrows():
        image_name = row["image_name"]
        account_name = row["source_account_name"]
        account_id = row["source_pf_account_id"]
        number_of_posts = row["number_of_retweets"]

        if image_name is None:
            pass

        if image_name not in results and image_name.lower() != "none":
            results[image_name] = {
                "accounts": [],
                "accounts_id": [],
                "number_of_posts": 0,
            }

        if image_name.lower() != "none":
            results[image_name]["accounts"].append(account_name)
            results[image_name]["accounts_id"].append(account_id)
            results[image_name]["number_of_posts"] += number_of_posts

    return results


reused_images_info = how_many_images_have_been_reused()
print(reused_images_info)

Des personnes publient 2 fois ou une plus la même images, d'autres part plusieurs comptes publient la même image.

# MA QUESTION : EST-CE QUE LE FAIT DE RETWEET SIGNIFIE QUE C"EST UN COMPOPRTEMENT INAUTHENTIQUE ? MEME SI L'IMAGE EST BIZARRE ?

In [ ]:
def convert_to_pd(data_dict):
    flattened_data = []
    for images_list, values in data_dict.items():
        for image in eval(images_list):
            flattened_data.append(
                {
                    "image_name": image,
                    "account": values["accounts"],
                    "account_id": values["accounts_id"],
                    "number_of_posts": values["number_of_posts"],
                }
            )
    data = pd.DataFrame(flattened_data)

    return data

In [ ]:
data = convert_to_pd(reused_images_info)

In [ ]:
data.describe

In [ ]:
data[data["account_id"].apply(len) > 1].head()

**On a des gens qui ont changé de nom au fil du temps. Regardez les 3 premières lignes. Ils diffusent les mêmes images avec des nom différents mais l'id reste identique.**

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Chemin du dossier contenant les images
image_dir = os.path.join(source_path, "img", "img")


def display_a_specific_img(img_name):
    img_path = os.path.join(image_dir, img_name)
    if not os.path.exists(img_path):
        print(
            f"Erreur : l'image {img_name} n'a pas été trouvée à l'emplacement {img_path}"
        )
        return
    img = mpimg.imread(img_path)

    plt.imshow(img)
    plt.axis("off")
    plt.show()

Cette images ci-dessous est une publiée par un compte qui a eu le nom `Nicolas qui paie`

In [ ]:
display_a_specific_img("Go6R1sPWwAAIKuA.jpg")

L'image ci-dessus est en lien fort avec `Nicolas qui paie` et a été publiée par plusieurs comptes (identifiants différents). Parcontre celle d'en bas a aussi été relayée mais elle n'a visiblement rien à voir avec notre fameux `Nicolas`.

In [ ]:
display_a_specific_img("GqVHS67XkAENMR0.jpg")

In [ ]:
import copy

df_nicolas_implied = copy.deepcopy(data)

In [ ]:
df_nicolas_implied["account"].unique()

In [ ]:
def plot_barplot(data, x_var_name, y_var_name, figsize=(10, 8)):
    data = data.reset_index(drop=True)
    print(data.columns)
    plt.figure(figsize=figsize)
    sns.barplot(x=x_var_name, y=y_var_name, data=data)
    plt.grid(True)
    plt.show()

In [ ]:
plot_barplot(data=data, x_var_name="image_name", y_var_name="number_of_posts")

## **UMAP : Clustering ou segmentation des données**

In [ ]:
data.describe()

In [ ]:
data_umap = copy.deepcopy(data)
data_umap = data_umap[data_umap["number_of_posts"] > 2]

In [ ]:
import umap
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
from adjustText import adjust_text  # Pour éviter les chevauchements

# Utilisation de la colonne 'image_name' et 'account_name' pour la vectorisation
data_umap["combined"] = (
    data_umap["image_name"]
    + " "
    + data_umap["account"].apply(lambda x: ", ".join(map(str, x)))
    + data_umap["account_id"].apply(lambda x: ", ".join(map(str, x)))
)


# Appliquer TF-IDF pour transformer les noms d'images et les comptes en vecteurs
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(data_umap["combined"])

# Appliquer UMAP pour réduire à 2 dimensions
umap_model = umap.UMAP(n_components=2)
umap_result = umap_model.fit_transform(vectors.toarray())

# Ajouter les coordonnées UMAP au DataFrame
data_umap["umap_x"] = umap_result[:, 0]
data_umap["umap_y"] = umap_result[:, 1]

# Visualisation des résultats
plt.figure(figsize=(20, 20))
plt.scatter(
    data_umap["umap_x"], data_umap["umap_y"], c="blue", label="Images & Accounts"
)

# Préparer les annotations
texts = []
for i, row in data_umap.iterrows():
    texts.append(
        plt.text(
            row["umap_x"],
            row["umap_y"],
            f"{row['image_name']} - {row['account']}",
            fontsize=7,
            alpha=0.7,
            ha="center",
            va="center",
        )
    )

# Ajuster les annotations pour éviter les chevauchements
adjust_text(texts, arrowprops=dict(arrowstyle="->", color="red", lw=0.5))

# Ajouter un titre et des labels
plt.title("UMAP Projection of Images and Accounts")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.show()

## LES IMAGES EN RAPPORT AVEC NICOLAS QUI PAIE

In [ ]:
import random

# Liste des fichiers d'image
image_to_plot = data["image_name"][
    data["account"].str.lower().str.contains("nicola", na=False)
]
image_to_plot = image_to_plot[:40]

# Nombre d'images à afficher
n_images = len(image_to_plot)

print(n_images)

In [ ]:
# Dimensions de la grille (ici 2 lignes et 3 colonnes, tu peux ajuster selon le nombre d'images)
ncols = 5
nrows = (n_images // ncols) + (n_images % ncols > 0)

# Créer une figure pour le plot
fig, axes = plt.subplots(nrows, ncols, figsize=(20, 20))

# Aplatir les axes en 1D pour faciliter l'itération
axes = axes.flatten()

# Afficher chaque image dans la grille
for i, image_file in enumerate(image_to_plot):
    img_path = os.path.join(image_dir, image_file)
    img = mpimg.imread(img_path)

    axes[i].imshow(img)
    axes[i].axis("off")  # désactiver les axes

# Supprimer les axes vides si le nombre d'images n'est pas un multiple exact de la grille
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

# aficher la figure
plt.tight_layout()
plt.show()